# Run All Evaluations

This notebook runs `evaluation.run_all_evals` instead of launching `eval_router`, `eval_NLU`, `eval_DM`, and `eval_NLG` one by one.

The project folder is found automatically in Google Drive by searching for project marker files. This works both for folders in `MyDrive` and for shared folders added as shortcuts to Drive.

Before running the full evaluation, make sure the project contains the latest versions of the evaluation scripts.


In [ ]:
from pathlib import Path
import os
import sys
import shutil
import subprocess

REPO_URL = "https://github.com/MessaAlberto/HMD_aquatic_center_chatbot.git"
PROJECT_DIR = Path("/content/HMD-project")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(
    ["git", "clone", REPO_URL, str(PROJECT_DIR)],
    check=True,
)

COLAB_UTILS_DIR = PROJECT_DIR / "colab_utils"
REQUIREMENTS_PATH = COLAB_UTILS_DIR / "requirements_colab.txt"

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.environ["REQUIREMENTS_PATH"] = str(REQUIREMENTS_PATH)

print("Project folder:", PROJECT_DIR)
print("Current directory:", os.getcwd())
print("Requirements file:", REQUIREMENTS_PATH)

In [ ]:
print("Installing requirements from:", REQUIREMENTS_PATH)
%pip install -q -r "$REQUIREMENTS_PATH"

In [ ]:
import os
import sys
import importlib
import logging

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("HF_TOKEN not found. Public models can still be loaded, but loading may fail.")

os.environ["APP_DEBUG"] = "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

importlib.invalidate_caches()

logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s:%(message)s",
    force=True,
)

print("Project configured.")

In [ ]:
from pathlib import Path

required_files = [
    "evaluation/run_all_evals.py",
    "evaluation/eval_router.py",
    "evaluation/eval_NLU.py",
    "evaluation/eval_DM.py",
    "evaluation/eval_NLG.py",
]

missing = [path for path in required_files if not Path(path).exists()]

if missing:
    print("Missing files:")
    for path in missing:
        print("-", path)
else:
    print("All evaluation files found.")

print("\nEvaluation folder:")
!ls -lah evaluation

## Select models and components

By default, the notebook tries to read all available models from `llm.config.MODELS`.

If you want a quick test, replace `MODELS_TO_EVALUATE` with a shorter list, for example:

```python
MODELS_TO_EVALUATE = ["qwen3_4b"]
```


In [ ]:
import importlib

try:
    config_module = importlib.import_module("llm.config")
    MODELS_TO_EVALUATE = list(config_module.MODELS.keys())
except Exception as exc:
    print("Could not automatically read llm.config.MODELS:", repr(exc))
    MODELS_TO_EVALUATE = ["qwen3_4b"]

COMPONENTS_TO_EVALUATE = ["router", "nlu", "dm", "nlg"]

# Lower values are safer on Colab when evaluating multiple models.
BATCH_SIZE = 4

print("Models to evaluate:", MODELS_TO_EVALUATE)
print("Components to evaluate:", COMPONENTS_TO_EVALUATE)
print("Batch size:", BATCH_SIZE)

## Run all evaluations

This cell launches:

```bash
python -m evaluation.run_all_evals --models ... --components router nlu dm nlg --batch-size ...
```

The script should load one model, evaluate all selected components, then move to the next model.


In [ ]:
import subprocess
import sys
import time
import shutil
import gc
from pathlib import Path

import torch


MODELS_TO_EVALUATE = [
    "qwen3_4b",
    "qwen25_3b",
    "gemma3_4b",
    "phi4_mini",
    "mistral7b",
]

MODEL_IDS = {
    "qwen3_4b": "Qwen/Qwen3-4B-Instruct-2507",
    "qwen25_3b": "Qwen/Qwen2.5-3B-Instruct",
    "gemma3_4b": "google/gemma-3-4b-it",
    "phi4_mini": "microsoft/Phi-4-mini-instruct",
    "mistral7b": "mistralai/Mistral-7B-Instruct-v0.3",
}


def print_disk_usage():
    print("\nDisk usage:")
    subprocess.run(["df", "-h", "/content"], check=False)
    subprocess.run(["du", "-sh", "/root/.cache/huggingface"], check=False)


def clean_model_cache(model_name: str):
    model_id = MODEL_IDS[model_name]
    cache_name = "models--" + model_id.replace("/", "--")

    paths_to_remove = [
        Path("/root/.cache/huggingface/hub") / cache_name,
        Path("/root/.cache/huggingface/modules"),
        Path("/root/.cache/torch"),
    ]

    for path in paths_to_remove:
        if path.exists():
            shutil.rmtree(path, ignore_errors=True)
            print("Removed:", path)

    gc.collect()

    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception as error:
            print("CUDA cleanup failed:", error)


all_results = {}

for model_name in MODELS_TO_EVALUATE:
    print("\n" + "=" * 100)
    print(f"Running evaluations for model: {model_name}")
    print("=" * 100)

    print_disk_usage()

    cmd = [
        sys.executable,
        "-m",
        "evaluation.run_all_evals",
        "--models",
        model_name,
        "--components",
        *COMPONENTS_TO_EVALUATE,
        "--batch-size",
        str(BATCH_SIZE),
    ]

    print("\nRunning command:")
    print(" ".join(cmd))

    start = time.time()

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()
    elapsed = time.time() - start

    all_results[model_name] = {
        "return_code": return_code,
        "elapsed_minutes": elapsed / 60,
    }

    print(f"\nFinished {model_name} in {elapsed / 60:.1f} minutes.")
    print("Return code:", return_code)

    print(f"\nCleaning cache for {model_name}...")
    clean_model_cache(model_name)

    print_disk_usage()

    if return_code != 0:
        print("\nLast 120 output lines:")
        print("=" * 100)
        print("".join(output_lines[-120:]))
        raise RuntimeError(f"run_all_evals failed for model: {model_name}")

print("\n" + "=" * 100)
print("All evaluations completed.")
print("=" * 100)

for model_name, result in all_results.items():
    print(
        f"{model_name}: return_code={result['return_code']}, "
        f"time={result['elapsed_minutes']:.1f} minutes"
    )

## Create a ZIP with all evaluation outputs

Run this at the end if you want to download the generated evaluation files from Colab.


In [ ]:
import re
import shutil
from pathlib import Path
from google.colab import files

def safe_name(value):
    value = str(value)
    value = re.sub(r"[^a-zA-Z0-9_.-]+", "_", value)
    return value.strip("_") or "unknown"

models_part = "_".join(safe_name(model) for model in MODELS_TO_EVALUATE)
components_part = "_".join(safe_name(component) for component in COMPONENTS_TO_EVALUATE)

zip_base = Path(f"evaluation_outputs__models_{models_part}__components_{components_part}")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir="evaluation")

print("Created:", zip_path)
files.download(zip_path)